# Fooocus on Google Colab

Select **Runtime → Change runtime type → GPU** and the current Python 3.13 runtime.

Run **Setup**, then **Start**. Fooocus uses Colab's installed GPU PyTorch instead of downloading an older CUDA stack. Setup reports the detected versions and stops on errors.

The application and active models run from `/content/Fooocus`. Drive keeps models under `MyDrive/Fooocus/models` and finished images under `MyDrive/Fooocus/outputs`. Existing Drive installations are retained. Put additional checkpoints in `models/checkpoints`, LoRAs in `models/loras`; they appear in the UI and are copied only when used.

The first run downloads the selected default model and required support models. Additional features download their models when first used. A second Start in the same session reuses local files; a new session copies needed models from Drive. Do not delete local output files until Drive saving has succeeded.

This notebook targets the current runtime; the exact GPU/Python package combination and a real GPU image generation still need to pass the documented Colab acceptance checks.


## Setup
Connect Drive, update your fork, and install only missing compatible dependencies.

In [ ]:
import subprocess, sys, time
from pathlib import Path

REPOSITORY = "https://github.com/abjilox/Fooocus.git"
BRANCH = "better-colab"
LOCAL_REPO = Path("/content/Fooocus")
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/Fooocus" #@param {type:"string"}
STATE_FILE = "/content/fooocus-colab-state.json"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

setup_started = time.monotonic()
def git(*args):
    return subprocess.check_output(["git", "-C", str(LOCAL_REPO), *args], text=True).strip()

if not LOCAL_REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, str(LOCAL_REPO)], check=True)
else:
    if not (LOCAL_REPO / ".git").exists():
        raise RuntimeError(f"{LOCAL_REPO} exists but is not a Git checkout. Choose a different local folder.")
    if git("remote", "get-url", "origin").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("The local checkout belongs to another repository. Keep it and choose a different LOCAL_REPO.")
    if git("branch", "--show-current") != BRANCH:
        raise RuntimeError(f"The local checkout is not on {BRANCH}. Choose a different local folder.")
    if git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError("Local source changes found. Save them before updating; no files were overwritten.")
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "merge", "--ff-only", "FETCH_HEAD"], check=True)

command = [sys.executable, str(LOCAL_REPO / "scripts/colab_bootstrap.py"), "prepare", "--state-file", STATE_FILE]
if USE_GOOGLE_DRIVE:
    command += ["--drive-root", DRIVE_ROOT]
subprocess.run(command, check=True)
print(f"Setup including repository update: {time.monotonic() - setup_started:.1f}s")


## Start
Run again to restart the application without repeating setup. Open the Gradio share URL printed below.

In [ ]:
import subprocess, sys

PRESET = "default" #@param ["default", "anime", "realistic"]
subprocess.run([sys.executable, str(LOCAL_REPO / "scripts/colab_bootstrap.py"),
                "start", "--state-file", STATE_FILE, "--preset", PRESET], check=True)
